# Differential Equations — Session 21
## Section 4.9: Systems by Elimination

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives


1. Write systems in differential-operator form.
2. eliminate one dependent variable.
3. solve the resulting scalar higher-order equation.
4. recover the eliminated variable and enforce coefficient relations.
5. compare elimination with numerical system integration.
6. interpret trajectories in the phase plane.


> The theoretical sequence follows the supplied section, while all examples, diagrams, and simulations are original.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |\n|---:|---|\n| 0–18 min | Operator notation and elimination |\n| 18–42 min | Worked homogeneous system |\n| 42–60 min | Recovering the second variable |\n| 60–78 min | IVP and phase-plane visualization |\n| 78–88 min | Matrix/exponential cross-check |\n| 88–90 min | Exit check |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE=True
except ImportError:
    WIDGETS_AVAILABLE=False

if os.environ.get('NB_VALIDATION_MODE') == '1':
    WIDGETS_AVAILABLE=False
plt.rcParams['figure.figsize']=(8,5)
plt.rcParams['axes.grid']=True
np.set_printoptions(precision=6,suppress=True)

def solve_second_order(f, span, y0, yp0, points=900, **kwargs):
    def rhs(x,z): return [z[1], f(x,z[0],z[1])]
    t=np.linspace(span[0],span[1],points)
    return solve_ivp(rhs,span,[y0,yp0],t_eval=t,**kwargs)

def wronskian_symbolic(funcs,x):
    return sp.simplify(sp.Matrix([[sp.diff(f,x,j) for f in funcs] for j in range(len(funcs))]).det())

print('Notebook ready.')
print('Interactive widgets available:',WIDGETS_AVAILABLE)

## Formal theory reference

A solution of a differential system is a tuple of functions satisfying every equation on a common interval.

For constant-coefficient operators, polynomial factors in $D$ commute. This permits algebraic elimination analogous to simultaneous algebraic equations.

After eliminating one variable, the scalar equation may contain more arbitrary constants than the original system permits. Substitution back into an original equation imposes relations among those constants.

### Classroom Checkpoint — Return to the Original System

After eliminating one dependent variable and solving the resulting higher-order equation, why must the solution be substituted back into the original system?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Example system

Consider

$$x'=x+2y,$$

$$y'=-2x+y.$$

Differentiate the first equation and eliminate $y,y'$ to obtain

$$x''-2x'+5x=0.$$

The same scalar equation holds for $y$.

In [ ]:
t=sp.symbols('t',real=True); A=sp.Matrix([[1,2],[-2,1]]); display(A.charpoly().as_expr()); display(A.eigenvals())

The roots $1\pm2i$ imply a growing rotation.

In [ ]:
def system_explorer(x0=1.0,y0=0.0,T=8):
    A=np.array([[1.,2.],[-2.,1.]])
    sol=solve_ivp(lambda t,z:A@z,(0,T),[x0,y0],t_eval=np.linspace(0,T,900))
    plt.plot(sol.t,sol.y[0],label='x'); plt.plot(sol.t,sol.y[1],label='y'); plt.legend(); plt.show()
    plt.plot(sol.y[0],sol.y[1]); plt.scatter([x0],[y0],s=70); plt.xlabel('x'); plt.ylabel('y'); plt.title('phase trajectory'); plt.show()
if WIDGETS_AVAILABLE: interact(system_explorer,x0=FloatSlider(min=-2,max=2,step=.25,value=1),y0=FloatSlider(min=-2,max=2,step=.25,value=0),T=IntSlider(min=2,max=12,step=1,value=8))
else: system_explorer()

## 2. Analytic recovery

From $x'=x+2y$,

$$y=\frac{x'-x}{2}.$$

Once $x$ is known, this formula automatically enforces the correct coefficient relations.

In [ ]:
t=sp.symbols('t',real=True); c1,c2=sp.symbols('c1 c2'); x=sp.exp(t)*(c1*sp.cos(2*t)+c2*sp.sin(2*t)); y=sp.simplify((sp.diff(x,t)-x)/2); display(y)

## 3. Matrix exponential cross-check

Although matrix methods come later, $e^{At}z_0$ provides a computational verification of the elimination solution.

In [ ]:
A=np.array([[1.,2.],[-2.,1.]]); z0=np.array([1.,.5]); tt=np.linspace(0,3,100); Z=np.array([expm(A*s)@z0 for s in tt]); plt.plot(tt,Z[:,0],label='x'); plt.plot(tt,Z[:,1],label='y'); plt.legend(); plt.show()

## Classroom Checkpoint — Exit Check

Eliminate $y$ from

$$x'=3x+y,\qquad y'=-4x+y.$$

> Pause here. Let students commit to an answer before running the next cell.